# Trois exercices de traitement

Sur des colonnes de notre fichier que **personne n'a encore analysées**.

La méthode est dans `docs/APPRENDRE-A-TRAITER.md`. Ici on la pratique.

**Règle du jeu :** chercher soi-même avant de dérouler la réponse, qui est
tout en bas.


## Mise en route

Une seule cellule à exécuter pour charger les données.


In [1]:
import csv
import re
from collections import Counter
from pathlib import Path

CHEMIN = Path("..") / "data" / "reference" / "drugs_40molecules.csv"
with open(CHEMIN, encoding="utf-8", newline="") as fh:
    lignes = list(csv.DictReader(fh))

print(f"{len(lignes)} produits, {len(lignes[0])} colonnes")
print()
for cle, valeur in lignes[0].items():
    print(f"  {cle:<20} {valeur}")


914 produits, 19 colonnes

  drug_id              DZ-0001
  therapeutic_class    ains
  inn                  ACIDE ACETYLSALICYLIQUE
  brand_name           ACEPRAL
  code                 03 A 001
  registration_no      003/03 A 001/97/20
  form                 COMPRIME
  strength_raw         500MG
  packaging_raw        B/20
  list_class           
  lab_name             GROUPE SAIDAL
  lab_country          ALGERIE
  date_initial         2020-08-13
  date_final           2020-08-13
  type_code            GE
  status               F
  stability            36 MOIS
  source_version       2026-08
  source_sheet         nomenclature


---

# Exercice 1 — la colonne `type_code`

Elle ne contient que `GE` et `RE`. Personne ne sait ce que ça veut dire : le
ministère ne publie aucune légende.

### 1.1 Commencer par compter

Toujours. Avant toute hypothèse.


In [2]:
compte = Counter(r["type_code"] for r in lignes)
for valeur, n in compte.most_common():
    print(f"  {valeur or '(vide)':<10} {n:>4}")


  GE          826
  RE           85
  (vide)        3


### 1.2 Poser une hypothèse

`GE` = **générique**, `RE` = **référence** ? C'est la lecture la plus
naturelle. Mais une lecture naturelle n'est pas une preuve.

### 1.3 Trouver comment la TESTER

C'est tout l'exercice. Il faut une conséquence **mesurable** de l'hypothèse.

> Si `GE` veut dire générique, alors un produit `GE` doit avoir beaucoup de
> « frères » : même substance, même dosage, des marques différentes. Un
> produit de référence devrait en avoir moins.

À toi d'écrire la mesure. La cellule suivante donne l'ossature.


In [3]:
import sys
sys.path.insert(0, str(Path("..") / "src"))
from pharmeazor.normalize.substance import parse_inn

# Combien de marques distinctes partagent la même substance ET le même dosage ?
familles = {}
for r in lignes:
    cle = (parse_inn(r["inn"]).cle(), r["strength_raw"])
    familles.setdefault(cle, set()).add(r["brand_name"])

# À toi : pour chaque type_code, la taille MOYENNE de la famille.
# (supprime le `pass` et écris la mesure)
pass


### 1.4 Conclure honnêtement

La vraie question de cet exercice n'est pas « quelle est la réponse ».

C'est : **l'écart mesuré est-il assez grand pour conclure ?**

Savoir répondre « non, pas assez » est plus difficile — et plus utile — que
de trouver une explication qui arrange.


---

# Exercice 2 — la colonne `code`

246 valeurs distinctes : `03 B 026`, `04 E 147`... Illisible tel quel.

### 2.1 Mesurer la FORME

La technique centrale du projet : remplacer chaque chiffre par `N`, puis
compter les motifs.


In [4]:
formes = Counter(re.sub(r"[0-9]", "N", r["code"]) for r in lignes)
for forme, n in formes.most_common(10):
    print(f"  {n:>4}  {forme!r}")


   359  'NN B NNN'
   245  'NN E NNN'
    94  'NN F NNN'
    81  'NN A NNN'
    61  'NN G NNN'
    27  'NN K NNN'
    15  'NN D NNN'
    11  'NN M NNN'
     9  'NN C NNN'
     8  'NN H NNN'


### 2.2 Ce que la forme révèle

Un seul motif domine, et il contient **une lettre au milieu**.

Une lettre au milieu d'un code, ça ne sort jamais de nulle part. Il faut la
croiser avec les autres colonnes pour voir à quoi elle correspond.


In [5]:
# À toi : extraire la lettre, puis la croiser avec `therapeutic_class`.
# Indice pour l'extraction :
#     re.match(r"^[0-9]{2}\s+([A-Z])\s", r["code"])
pass


### 2.3 Conclure — et surtout, avec quelle prudence

Tu vas trouver un signal net sur certaines lettres, et confus sur d'autres.

La bonne réponse n'est pas « la lettre = la classe ». C'est une phrase qui dit
**ce qui est mesuré, ce qui ne l'est pas, et ce qu'on en fait** — c'est-à-dire
rien, dans les données.


---

# Exercice 3 — le `registration_no`

`077/03 B 026/17/23`. Plusieurs morceaux, séparés par des `/`.

### 3.1 Mesurer sa forme


In [6]:
formes = Counter(re.sub(r"[0-9]", "N", r["registration_no"]) for r in lignes)
print(f"  {len(formes)} motifs distincts")
for forme, n in formes.most_common(8):
    print(f"  {n:>4}  {forme!r}")


  22 motifs distincts
   209  'NNN/NN B NNN/NN/NN'
   150  'NNN/NN B NNN/NN'
   146  'NNN/NN E NNN/NN/NN'
    99  'NNN/NN E NNN/NN'
    51  'NNN/NN F NNN/NN/NN'
    44  'NNN/NN A NNN/NN/NN'
    43  'NNN/NN F NNN/NN'
    38  'NNN/NN G NNN/NN/NN'


### 3.2 Isoler le premier morceau

Les trois premiers chiffres. Que peuvent-ils bien désigner ?

Pour le savoir : les croiser avec chaque autre colonne et regarder laquelle
« colle ».


In [7]:
# À toi : pour chaque préfixe de 3 chiffres, combien de laboratoires distincts ?
# Si la réponse est « un seul » dans la grande majorité des cas, le préfixe
# identifie quelque chose de précis.
pass


### 3.3 Et les exceptions ?

Il en restera quelques-unes. Deux attitudes possibles :

- les ignorer parce qu'elles gênent — **non**
- les regarder, et chercher si une autre donnée du projet les explique

Indice : regarde `data/reference/desaccords_versions_40molecules.csv`, la
colonne `laboratoire`. Un nom de laboratoire peut changer entre deux versions
du registre sans que le laboratoire change.

Même là : une explication plausible n'est pas une source.


---

# Réponses

À ne dérouler qu'après avoir essayé. Elles sont dans
`docs/APPRENDRE-A-TRAITER.md`, section « Réponses des exercices ».

Les cellules ci-dessous donnent les mesures. **Lire d'abord la section du
document** : elle explique *comment formuler* la conclusion, ce qui est la
vraie difficulté.


In [8]:
# RÉPONSE 1
tailles = {}
for r in lignes:
    cle = (parse_inn(r["inn"]).cle(), r["strength_raw"])
    tailles.setdefault(r["type_code"], []).append(len(familles[cle]))

for type_code, valeurs in sorted(tailles.items()):
    moyenne = sum(valeurs) / len(valeurs)
    print(f"  {type_code or '(vide)':<10} famille moyenne : {moyenne:.1f} marques")

print()
print("  L'écart va dans le sens attendu, mais il est FAIBLE.")
print("  7,9 contre 5,6 ne suffit pas à trancher : hypothèse, pas conclusion.")


  (vide)     famille moyenne : 1.0 marques
  GE         famille moyenne : 7.9 marques
  RE         famille moyenne : 5.6 marques

  L'écart va dans le sens attendu, mais il est FAIBLE.
  7,9 contre 5,6 ne suffit pas à trancher : hypothèse, pas conclusion.


In [9]:
# RÉPONSE 2
lettres = {}
for r in lignes:
    m = re.match(r"^[0-9]{2}\s+([A-Z])\s", r["code"])
    if m:
        lettres.setdefault(m.group(1), Counter())[r["therapeutic_class"]] += 1

for lettre in sorted(lettres):
    detail = ", ".join(f"{k} {v}" for k, v in lettres[lettre].most_common(3))
    print(f"  {lettre}  {detail}")

print()
print("  G ne contient QUE des antibiotiques, E presque que des")
print("  antihypertenseurs. Mais B mélange quatre classes : ce n'est donc")
print("  pas une classification stricte. Mesuré, non documenté, rien écrit")
print("  dans les données.")


  A  ains 44, antibiotique 24, antalgique 13
  B  ains 174, antalgique 119, antibiotique 39
  C  antibiotique 6, antalgique 3
  D  antibiotique 15
  E  antihypertenseur 203, antibiotique 41, ains 1
  F  betabloquant 39, antalgique 29, antibiotique 26
  G  antibiotique 61
  H  diuretique 7, antalgique 1
  J  ains 4
  K  antibiotique 27
  M  antibiotique 8, antihypertenseur 3

  G ne contient QUE des antibiotiques, E presque que des
  antihypertenseurs. Mais B mélange quatre classes : ce n'est donc
  pas une classification stricte. Mesuré, non documenté, rien écrit
  dans les données.


In [10]:
# RÉPONSE 3
prefixes = {}
for r in lignes:
    m = re.match(r"^([0-9]{3})", r["registration_no"])
    if m:
        prefixes.setdefault(m.group(1), set()).add(r["lab_name"])

seuls = sum(1 for labos in prefixes.values() if len(labos) == 1)
print(f"  {len(prefixes)} préfixes distincts")
print(f"  {seuls} sur {len(prefixes)} ne correspondent qu'à UN laboratoire")
print(f"  soit {100 * seuls / len(prefixes):.0f} %")
print()
print("  Très net : le préfixe est presque certainement un code laboratoire.")
print("  Les exceptions s'expliquent bien par un changement de raison sociale")
print("  — LAM devenu LABORATOIRE ALGERIEN DU MEDICAMENT entre deux versions.")
print("  Mais « s'explique bien » n'est pas une source : hypothèse mesurée,")
print("  avec ses exceptions, et rien écrit dans les données.")


  121 préfixes distincts
  112 sur 121 ne correspondent qu'à UN laboratoire
  soit 93 %

  Très net : le préfixe est presque certainement un code laboratoire.
  Les exceptions s'expliquent bien par un changement de raison sociale
  — LAM devenu LABORATOIRE ALGERIEN DU MEDICAMENT entre deux versions.
  Mais « s'explique bien » n'est pas une source : hypothèse mesurée,
  avec ses exceptions, et rien écrit dans les données.


---

# Ce que les trois exercices ont en commun

Dans les trois cas, la mesure donne un **signal**, jamais une **certitude**.

| | signal | ce qu'on en fait |
|---|---|---|
| `type_code` | faible (7,9 contre 5,6) | on ne conclut pas |
| `code` | net sur `G` et `E`, confus sur `B` | hypothèse documentée |
| `registration_no` | très net (92 %) | hypothèse documentée, avec ses exceptions |

**Aucun des trois ne se traduit par une ligne écrite dans les données.**

C'est le cœur du métier : mesurer beaucoup, écrire peu, et toujours savoir
dire lequel des deux on est en train de faire.
